# Data Analysis (Part 6) - Regression - CLASS EXERCISE SOLUTIONS

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import statsmodels.api as sm

pd.options.display.float_format = '{:,.3f}'.format # Display option to include 2 decimal places on all floats
pd.set_option('display.max_columns', None) # Display option to show all columns
pd.set_option('display.max_rows', None) # Display option to show all rows

#### Exercise 1

You have been provided with a dataset called `insurance.csv` which tracks medical costs billed by a health insurance company along with several demographic and other variables. The data was obtained from https://www.kaggle.com/mirichoi0218/insurance. The following are the variable definitions:

|Variable|Definition|
| --- | --- |
|age|Age|
|sex|female or male|
|bmi|Body mass index|
|children|Number of children covered by health insurance|
|smoker|Smoker - yes/no|
|region|The beneficiary's residential area in the US - northeast, southeast, southwest, northwest|
|charges|Individual medical costs billed by health insurance|
    
1. Read the `insurance.csv` file into a pandas DataFrame called `df` and create the following columns:

    1. `male`: equal to 1 if `sex` is equal to `'male'` and equal to 0 if `sex` is equal to `'female'`.
    
    2. `smoke`: equal to 1 if `smoker` is equal to `'yes'` and equal to 0 if `smoker` is equal to `'no'`.

**HINT**:

        df['new_column'] = np.where(CONDITIONAL_STATEMENT, VALUE_IF_TRUE, VALUE_IF_FALSE)

In [ ]:
# 1

df = pd.read_csv('insurance.csv')

df['male'] = np.where(df.sex == 'male', 1, 0)

df['smoke'] = np.where(df.smoker == 'yes', 1, 0)

df.head()

2. Print the summary statistics of the numerical columns in `df`. What is the average age? What is the percentage of observations that are male? What is the percentage of observations that are smokers?

In [ ]:
# 2

df.describe().transpose()

# Average age: 39.2 years
# % that are male: 50.5%
# % that are smokers: 20.5%

3. Run a linear regression with `charges` as the dependent variable and the following independent variables: `age`, `bmi`, `children`, `male`, and `smoke`.

In [ ]:
# 3

X = df[['age','bmi','children','male','smoke']]
y = df['charges']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

4. Interpret the regression output:

    1. Which variables are significantly related to insurance charges? What do the coefficient estimates mean economically? (e.g., for a one unit increase in age, charges increases by $X)
    
    2. What is the `Adjusted R-squared`? What does this number mean?

In [ ]:
# 4

# All of the independent variables are significant at the 5% level or lower except for 'male'.

# Each of the significant variables is positively related to insurance charges.

# For example, the coefficient on 'age' is equal to 257.74 which means that for each additional year of age,
# insurance costs increase by $257.74.

# Another example, the coefficient on 'smoke' is equal to 2.382e+04 which is shorthand for 23,820
# which means that if the person is a smoker, insurances chargers are $23,820 higher.

# The adjusted R-squared is equal to 0.749 which suggests that the combination of these independent variables explains
# 74.9 percent of the variation in insurance charges.

#### Exercise 2

Run the cell below to read in the `insurance.csv` file and run the regression model we used in Exercise 1. You will use this data to examine excess insurances charges by region.

In [ ]:
df = pd.read_csv('insurance.csv')
df['male'] = np.where(df.sex == 'male', 1, 0)
df['smoke'] = np.where(df.smoker == 'yes', 1, 0)
X = df[['age','bmi','children','male','smoke']]
y = df['charges']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

1. Use `df` to create a bar chart of the average charges by region. Which region has the highest average insurance charges?

In [ ]:
# 1

avg_charges = df.groupby('region')['charges'].mean().reset_index()
avg_charges = avg_charges.sort_values(by='charges')
display(avg_charges)

plt.figure(figsize=(10,6))
plt.bar(avg_charges['region'],avg_charges['charges'])
plt.xlabel('Region')
plt.ylabel('Avg. Insurance Charges')
plt.title("Avg. Insurance Charges by Region")
plt.show()

# The southeast region has the highest average insurance charges.

2. Using our linear `model`, create a column called `excess_charges` in `df` which represents the residual value of `charges`. Then create a bar chart of the average excess charges by region. Which region has the highest average excess insurance charges? What does this mean?

In [ ]:
# 2

df['charges_hat'] = model.predict(X)
df['excess_charges'] = df['charges'] - df['charges_hat']

avg_excess_charges = df.groupby('region')['excess_charges'].mean().reset_index()
avg_excess_charges = avg_excess_charges.sort_values(by='excess_charges')
display(avg_excess_charges)

plt.figure(figsize=(10,6))
plt.bar(avg_excess_charges['region'],avg_excess_charges['excess_charges'])
plt.xlabel('Region')
plt.ylabel('Avg. Excess Insurance Charges')
plt.title("Avg. Excess Insurance Charges by Region")
plt.show()

# The northeast region has the highest excess insurance charges. This means that after controlling
# for factors like age, bmi, children, gender, and smoking, those in the northeast region tend to 
# have more charges than those in other regions. Perhaps these individuals demand more health care
# than those in other regions, something our model does not take into account.

#### Exercise 3

Run the cell below to read in the `insurance.csv` file we used in Exercise 1. You will use this data to create out-of-sample predictions for insurance charges.

In [ ]:
df = pd.read_csv('insurance.csv')
df['male'] = np.where(df.sex == 'male', 1, 0)
df['smoke'] = np.where(df.smoker == 'yes', 1, 0)

1. Using `df`, create a training set called `train` containing a random subsample of 70% of the observations in `df`. Set the `random_state` in the `sample` function equal to `73723`. Create a test set called `test` containining the observations in `df` that are not in `train`.

In [ ]:
# 1

# Create the training set

train = df.sample(frac=0.7, random_state=73723)

# All remaining rows go into the test set

test=df.drop(train.index)

2. Run a linear regression model using the `train` DataFrame with `charges` as the dependent variable and the following independent variables: `age`, `bmi`, `children`, `male`, and `smoke`.

In [ ]:
# 2

X = train[['age','bmi','children','male','smoke']]
y = train['charges']

X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

3. Create a variable in the `test` DataFrame called `charges_hat` equal to the `charges` prediction obtained by applying the model created in the previous step to the observations in the `test` DataFrame.

In [ ]:
# 3

X_test = test[['age','bmi','children','male','smoke']]
X_test = sm.add_constant(X_test)

test['charges_hat'] = model.predict(X_test)
test.head(10)

4. Evaluate the `charges_hat` prediction by estimating a simple linear regression with `charges` as the dependent variable and `charges_hat` as the independent variable in the `test` DataFrame. Comment on the model's ability to predict charges in the out-of-sample test set.

In [ ]:
# 4

X = test[["charges_hat"]] 
y = test["charges"]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

# The adjusted R-squared is equal to 0.786 which is pretty high suggesting that the model
# does a decent job of predicting charges in the out-of-sample data.

# The coefficient on charges_hat is equal to 1.0364 which is close to 1 suggesting that charges_hat
# is a close approximation of the actual charges.

5. Visually inspect the relationship between `charges` and `charges_hat` by creating a `matplotlib` scatter plot with `charges` on the y-axis and `charges_hat` on the x-axis.

In [ ]:
# 5

plt.figure(figsize=(10,5))
plt.scatter(test['charges_hat'],test['charges'])
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.xlabel('Predicted Charges',fontsize=15)
plt.ylabel('Actual Charges',fontsize=15)
plt.title('Compare Actual Charges to Out-of-Sample Predictions',fontsize=20)
plt.show()